# Tutorial 02: Your First PINN

In the previous tutorial, we saw that an MLP trained on $[-2\pi, 0]$ fits $\sin(x)$ well within that range but fails completely on $[0, 2\pi]$ — the network has no reason to produce a sine wave where it has never seen data.

**The key insight of PINNs**: we know that $y = \sin(x)$ satisfies the ODE

$$y' = \cos(x), \quad y(0) = 0.$$

If we embed this ODE into the loss function, the network is forced to obey the physics *everywhere* — even in regions without training data. This is the core idea behind **Physics-Informed Neural Networks (PINNs)**: encoding known equations as soft constraints so the model generalizes beyond the observed domain.

#### How it works

A standard neural network minimizes a **data loss** over labeled samples:

$$\mathcal{L}_{\text{data}} = \frac{1}{N}\sum_{i=1}^{N} \left( \hat{y}(x_i) - y_i \right)^2$$

A PINN adds a **physics loss** that penalizes violation of the governing equation at a set of **collocation points** $\{x_j^c\}$ (which need no labels):

$$\mathcal{L}_{\text{physics}} = \frac{1}{M}\sum_{j=1}^{M} \left( \frac{d\hat{y}}{dx}\bigg|_{x_j^c} - \cos(x_j^c) \right)^2$$

The total loss is simply their sum:

$$\mathcal{L} = \mathcal{L}_{\text{data}} + \mathcal{L}_{\text{physics}}$$

The derivative $d\hat{y}/dx$ is computed exactly via **automatic differentiation** (the same backpropagation machinery used for training). No finite differences needed.

#### Implementation in Talos

In Talos, the trainer computes $\mathcal{L}_{\text{data}}$ as usual. The model can inject $\mathcal{L}_{\text{physics}}$ by overriding `model_loss(X, outputs, Y)` — the trainer automatically adds the two together. This keeps the physics knowledge encapsulated in the model where it belongs.

### Step 0: Setups
---

In [ ]:
# To allow automatically uses the newest version of your code.  
%load_ext autoreload
%autoreload 2

In [ ]:
# Add necessary paths to the system path
from utils import add_necessary_paths
add_necessary_paths()

# Import libraries
from utils import check_torch

import utils.u01 as u
import talos as ta
import numpy as np

# Do sanity checks before running the code
check_torch()

# Fix random seed for reproducibility
# NOTE: Run cells IN ORDER to ensure deterministic behavior.
ta.set_seed()

### Step 1: Prepare data
---

We use the exact same setup that failed in Tutorial 01: 200 points over $[-2\pi, 2\pi]$, split 50/50 **without shuffling**. This gives us:

- **Train set**: the left half $[-2\pi, 0]$ — the only region with labeled data
- **Test set**: the right half $[0, 2\pi]$ — the network must extrapolate here

A plain MLP failed catastrophically on the test set. Let's see if adding physics helps.

In [ ]:
# configure data set  
t_min, t_max = -2 * np.pi, 2 * np.pi
n_points = 200
shuffle = False
train_test_ratio = (1, 1)

# Generate data
X = np.linspace(t_min, t_max, n_points).reshape(-1, 1)  # Reshape to a column vector
Y = np.sin(X).reshape(-1, 1)  # Reshape to a column vector
dataset = ta.Dataset(X, Y, name='sin(x)')
dataset.report()

# Split data
train_set, test_set = dataset.split(*train_test_ratio, shuffle=shuffle)
train_set.report()
test_set.report()

# Visualize data
u.plot_data(train_set, test_set, title=f'Dataset Visualization (shuffle = {shuffle})')

### Step 2: Define the PINN model
---

To build a PINN, we subclass `MLP` and override a single method: `model_loss(X, outputs, Y)`. The trainer calls this automatically after computing the data loss and adds the result to the total loss.

Inside `model_loss`, we:
1. **Sample collocation points** uniformly over $[-2\pi, 2\pi]$ — these require no labels
2. **Forward pass** the collocation points through the network to get $\hat{y}$
3. **Compute $d\hat{y}/dx$** via `torch.autograd.grad` (exact automatic differentiation)
4. **Return the ODE residual** $\text{mean}(\hat{y}' - \cos x)^2$

Note that collocation points cover the *entire* domain, including $[0, 2\pi]$ where we have no data. This is how the physics loss guides the network in unseen regions.

In [ ]:
import torch

class SinPINN(ta.model.torch_zoo.MLP):
  """MLP with a physics-informed loss for y' = cos(x)."""

  def __init__(self, *args, n_collocation=200, x_min=-2*np.pi, x_max=2*np.pi,
               **kwargs):
    super().__init__(*args, **kwargs)
    self.n_collocation = n_collocation
    self.x_min = x_min
    self.x_max = x_max

  def model_loss(self, X, outputs, Y):
    """Physics loss: penalize violation of y' = cos(x) at collocation points."""
    # (1) Sample collocation points over the full domain.
    x_c = (torch.rand(self.n_collocation, 1, device=X.device)
           * (self.x_max - self.x_min) + self.x_min)
    x_c.requires_grad_(True)

    # (2) Forward pass at collocation points.
    y_c = self.forward(x_c)

    # (3) Compute dy/dx via autograd.
    dy_dx = torch.autograd.grad(
      y_c, x_c, grad_outputs=torch.ones_like(y_c), create_graph=True)[0]

    # (4) ODE residual: y' - cos(x).
    residual = dy_dx - torch.cos(x_c)
    return torch.mean(residual ** 2)

### Step 3: Train and evaluate
---

We use the same architecture and training setup as Tutorial 01 — the only difference is that `SinPINN` replaces `MLP`. The trainer doesn't need any changes: it detects the model's `model_loss` and adds it to the data loss automatically.

After training, we evaluate on the **right half** $[0, 2\pi]$ — the region where the plain MLP failed.

In [ ]:
# Model configuration
hidden_features = [50, 50, 20, 50, 50]

# Initialize PINN model
pinn = SinPINN(1, hidden_features, 1, activation='tanh')

# Train on the left half only (same setup as tutorial 01's failure case)
trainer = u.get_trainer(pinn, loss_fn='mse', early_stop=True, patience=2,
                        val_ratio=0.1, validate_every=10000, print_every=5000)

trainer.train(train_set, 50000)

In [ ]:
# Evaluate on the right half (unseen during training)
Y_pred = pinn.predict(test_set)
u.plot_data(train_set, test_set, Y_pred, eval=True,
            title='PINN Prediction vs Ground Truth (Test Set)')

### Takeaway

By adding a single `model_loss` method — just ~15 lines of code — we transformed a plain MLP into a PINN that can extrapolate beyond its training data. The key ingredients were:

- **Collocation points** sampled over the full domain (no labels needed)
- **Automatic differentiation** to compute $d\hat{y}/dx$ exactly
- **ODE residual** as a soft constraint added to the data loss

The network now learns not just from data, but from the *law* that governs the data. This is the power of physics-informed learning: even with limited observations, the model generalizes because it respects the underlying equations.